# 01 — Fixed-split TBI mouse training

Train 3DR-Net or the existing standard 3D U-Net on an explicit animal-level split.
The shared experiment core owns loading, preprocessing, compilation, checkpoint selection,
and optional held-out full-volume evaluation. Run with the **rnet** environment.

Start by reviewing the configuration and split counts. Set `RUN_TRAINING = True` when ready.
Leave `RESUME_RUN=None` for a fresh experiment; set it to resume a run with matching saved configuration and splits.
The historical notebooks remain references. No architectural defaults or dropout rates are changed.

## 1. Workspace and runtime
Configure GPU memory and precision before constructing generators or models.

In [1]:
from pathlib import Path
import sys

REPO_ROOT: Path = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "src/experiments/config.py").is_file()
)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.experiments.training import configure_runtime

TRAINING_SEED: int = 42
runtime = configure_runtime(seed=TRAINING_SEED, memory_growth=True, mixed_precision=False)
runtime
import nibabel as nib

REFERENCE: Path = (
    REPO_ROOT / "example" / "RARE" / "TBI_fm_19_49"
    / "Anat" / "TBI_fm_19_49_N4.nii.gz"
)
ref_img = nib.load(REFERENCE)
reference_axes = nib.aff2axcodes(ref_img.affine)
print("Reference axes:", reference_axes, "(left-right is the R/L axis)")


{'tensorflow': '2.10.0',
 'gpu_devices': ['/physical_device:GPU:0'],
 'precision': 'float32',
 'seed': 42}

## 2. Experiment configuration

The default task uses D1, with left/right lateral ventricles merged into one class,
as in the historical training notebook. D1 also contains fully annotated scans;
`unknown="background"` explicitly projects their additional anatomy labels into task background. Normalized metadata supplies annotation availability.
Reorientation uses the frozen reference below, before all other preprocessing.
Existing N4 images are loaded; bias correction is not repeated.

All stages resample onto an orthogonal grid before augmentation, removing source affine shear
while retaining the physical field of view. Images use linear interpolation and masks use nearest neighbour. Validation uses a fixed
centre crop and no augmentation. Full-volume testing does not crop the input.
`random_crop=True, crop_mode="center"` uses centre-biased random training crops with `std=None` by default. Use `crop_mode="random"` for uniform sampling or `random_crop=False` for fixed centre crops.

For partial labels, select `dice_ce_gradient_fix` and explicitly configure the complete source
mapping and class names. The core trains ROI logits and exports softmax inference probabilities.
Legacy sources require an explicit modality and `available_labels` in `DatasetSpec`.

Full-volume outputs use `original_label_ids=True`. `postprocessing_settings` controls brain-mask cleanup and ventricle separation. Thresholds are voxel counts on the 0.1 mm grid; categorical ROI masks are not cleaned as binary masks. Merged D1/rat ventricles are separated before mapping to original IDs 3 (contra), 13 (ipsi), and 21 (third ventricle). MouseFull keeps its directly predicted sides. GT uses the selected original annotations, never the prediction postprocessor.

The historical `default_hemisphere="right"` selects the plane side containing array origin, rather than deriving anatomical right from the affine. It is used when no sufficiently large lesion is predicted. Splitting requires at least three predicted third-ventricle voxels; missing midline predictions raise an explicit scan-specific error.


In [ ]:
from src.experiments.config import (
    DataConfig, DatasetSpec, ExperimentConfig, FixedSplitConfig, InferenceConfig,
    LabelConfig, LossConfig, ModelConfig, PostprocessingConfig, PreprocessingConfig,
)
from src.evaluation.k_fold import TrainingConfig

DATA_ROOT: Path = Path("D:/Other files/TESI_CNN/MANUSCRIPT_DATASET/normalized")
RUN_TRAINING: bool = True
RESUME_RUN: Path | None = None
SPLIT_FILE: Path | None = None

training = TrainingConfig(
    model_name="d1_3drnet", epochs=300, batch_size=8, seed=TRAINING_SEED, data_multiplier=3,
    output_dir=str(REPO_ROOT / "results"), monitor="val_loss", monitor_mode="min", default_callbacks=False,
    fit_kwargs={"verbose": 1},
)
# Fractions of all animals; saved split files keep their exact memberships.
split_settings = FixedSplitConfig(
    validation_fraction=0.2, test_fraction=0.1,
)

data_settings = DataConfig(root=DATA_ROOT, datasets=(DatasetSpec(name="D1", annotation_scope="full_volume"),))
preprocessing = PreprocessingConfig(
    reference=REFERENCE, patch_size=(80, 80, 80), spacing_mm=(0.1, 0.1, 0.1),
    correct_units=True, augment=True, random_crop=True, crop_mode="center",
    flip_probability=0.5,
)
model_settings = ModelConfig(architecture="3drnet", filters=(16, 32, 64, 128, 256))
loss_settings = LossConfig(regions="dice_ce", gamma=0.8, brain_alpha=0.6, batch_wise=True)
# Full volumes: postprocess first, then restore original annotation IDs.
postprocessing_settings = PostprocessingConfig(
    clean_brain_mask=True,
    fill_small_holes=True, holes_max_area=20000,
    remove_small_objects=True, object_min_area=30000,
    split_ventricles=True, use_centroids=False,
    default_hemisphere="right", lesion_voxels_threshold=10,
)
inference_settings = InferenceConfig(
    enabled=True, stride=40, save_predictions=True,
    original_label_ids=True, postprocessing=postprocessing_settings,
)

config = ExperimentConfig(
    data=data_settings, preprocessing=preprocessing, training=training,
    fixed_split=split_settings,
    model=model_settings, labels=LabelConfig(unknown="background"), loss=loss_settings,
    inference=inference_settings, learning_rate=0.001, split_seed=42,
)
config.validate()

## Configure callbacks directly

Fixed-split training uses the original LR reduction, TensorBoard histograms, early stopping and CSV logging. Edit the callback constructors or the list in the next cell; the training call explicitly receives `callbacks=callbacks`. No callback is added by a hidden experiment factory. `TrainingConfig.default_callbacks=False` prevents duplicate automatic logging/LR callbacks. Checkpoint selection uses the visible `TrainingConfig.monitor` and `monitor_mode` settings.

Relative log paths resolve inside each run/fold, so folds and study variants do not share files. The runner copies unused callback instances for every fit and saves their settings in `callbacks.json`; resume requires the same saved settings. CSV logs append when resuming.

The runner always saves best weights and epoch recovery state, even when this list is empty. It restores the selected best checkpoint before evaluation. For that reason early stopping uses `restore_best_weights=False`: restoring weights inside its epoch-end hook would mix earlier weights with the latest optimizer in the recovery checkpoint. The final best-weight behavior matches the original workflow.


In [ ]:
import tensorflow as tf

# Relative logging paths are resolved inside this run's fold directory.
log_dir: str = "tensorboard"
csv_filename: str = "training.csv"
csv_logger = tf.keras.callbacks.CSVLogger(filename=csv_filename, append=True)

callbacks: list[tf.keras.callbacks.Callback] = [
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", mode="min", factor=0.5, patience=10,
        min_lr=1e-7, verbose=1,
    ),
    tf.keras.callbacks.TensorBoard(log_dir=log_dir, histogram_freq=1),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", mode="min", patience=20, verbose=1,
        restore_best_weights=False,  # The runner reloads the best saved weights before evaluation.
    ),
    csv_logger,
]


## 3. Load datasets and select animals

Provide an existing `SPLIT_FILE` for matched or predefined experiments. Otherwise this cell
creates a reproducible, unstratified split using fractions of the complete animal cohort.
Every selected animal and all its timepoints belong to exactly one role.
Review cohort composition before training; D2 needs a separately designed small-cohort protocol.
The exact split and original-ID manifest are saved with the run.

In [3]:
from IPython.display import display
from src.experiments.data import SubjectSplit, prepare_data

prepared = prepare_data(config)

if RESUME_RUN is not None:
    saved_split = RESUME_RUN / "splits.json"
    split = SubjectSplit.load(saved_split, prepared.subjects)
    if SPLIT_FILE is not None:
        requested_split = SubjectSplit.load(SPLIT_FILE, prepared.subjects)
        if requested_split != split:
            raise ValueError("SPLIT_FILE differs from the resumed run's split")
elif SPLIT_FILE is None:
    split = prepared.make_split()
else:
    split = SubjectSplit.load(SPLIT_FILE, prepared.subjects)

display(prepared.split_summary(split))
display(prepared.subjects.manifest(list(split.train)).head())

,Role,Animals,Scans
0,train,77,77
1,validation,23,23
2,test,12,12


,Subject,Group,Scan,Dataset,SubjectID,OriginalID,Timepoint,Modality,ImagePath
0,D1/SUBJ_0002,D1_T1w,SUBJ_0002,D1,SUBJ_0002,Isch_PAL_PPIAKO_1777,None,T1w,D:\Other files\TESI_CNN\MANUSCRIPT_DATASET\nor...
1,D1/SUBJ_0004,D1_T1w,SUBJ_0004,D1,SUBJ_0004,Isch_PAL_PPIAKO_1782,None,T1w,D:\Other files\TESI_CNN\MANUSCRIPT_DATASET\nor...
2,D1/SUBJ_0006,D1_T1w,SUBJ_0006,D1,SUBJ_0006,Isch_PAL_PPIAKO_1784,None,T1w,D:\Other files\TESI_CNN\MANUSCRIPT_DATASET\nor...
3,D1/SUBJ_0007,D1_T1w,SUBJ_0007,D1,SUBJ_0007,Isch_PAL_PPIAKO_1785,None,T1w,D:\Other files\TESI_CNN\MANUSCRIPT_DATASET\nor...
4,D1/SUBJ_0009,D1_T1w,SUBJ_0009,D1,SUBJ_0009,Isch_PAL_PPIAKO_1790,None,T1w,D:\Other files\TESI_CNN\MANUSCRIPT_DATASET\nor...


## 4. Check one validation batch

This optional preparation check loads one batch without training a model. Validation retains
its final incomplete batch and never repeats scans. Full-volume manuscript metrics are distinct
from these centre-crop training diagnostics.

In [4]:
validation_generator = prepared.make_generator(split.validation, training=False, batch_size=1)
images, targets = validation_generator[0]

print("Input:", images.shape)
print("ROI targets:", targets[0].shape)
print("Brain targets:", targets[1].shape)
print("Validation batches:", len(validation_generator))

del images, targets, validation_generator

Generator configuration:
- Dataset path: D:\Other files\TESI_CNN\MANUSCRIPT_DATASET\normalized
- Dimensions: (80, 80, 80)
- Batch size: 1
- Number of input channels: 1
- Number of classes: 4
- Shuffle: False

Input: (1, 80, 80, 80, 1)
ROI targets: (1, 80, 80, 80, 4)
Brain targets: (1, 80, 80, 80, 1)
Validation batches: 23


## 5. Train and evaluate

The runner restores the best validation checkpoint before optional held-out test inference.
Test results are not used for checkpoint selection. Each execution creates a separate run folder.
Prediction files restore the original image grid and **original annotation IDs**.
Merged ventricles are separated with the historical postprocessor; the brain mask is cleaned separately.
The explicit settings and exported label mappings are saved with the evaluation.


In [5]:
from src.experiments.training import run_training

result = None
if RUN_TRAINING:
    result = run_training(prepared, split, resume_run=RESUME_RUN, callbacks=callbacks)
    print("Run saved to:", result.run_dir)
else:
    print("Configuration checked. Set RUN_TRAINING = True and rerun this cell to train.")

Generator configuration:
- Dataset path: D:\Other files\TESI_CNN\MANUSCRIPT_DATASET\normalized
- Dimensions: (80, 80, 80)
- Batch size: 8
- Number of input channels: 1
- Number of classes: 4
- Shuffle: True

Generator configuration:
- Dataset path: D:\Other files\TESI_CNN\MANUSCRIPT_DATASET\normalized
- Dimensions: (80, 80, 80)
- Batch size: 8
- Number of input channels: 1
- Number of classes: 4
- Shuffle: False

Epoch 1/300


KeyboardInterrupt: 

## 6. Inspect results
Plotly uses the native VS Code renderer; no widgets or embedded HTML are needed.

In [ ]:
import pandas as pd
import plotly.io as pio
from src.experiments.training import history_figure

pio.renderers.default = "vscode"

if result is not None:
    figure = history_figure(result)
    figure.show()
    display(pd.DataFrame([result.summaries]))

    if result.evaluations.get("regions"):
        metrics = pd.concat(result.evaluations["regions"], ignore_index=True)
        dice = metrics.loc[metrics["Metric"] == "Dice"]
        display(dice[["Subject", "Scan", "Region", "Value", "Status"]])

## Saved artifacts and next steps

The run contains `experiment.json`, `metadata.json`, `subjects.csv`, and `splits.json`.
`fold_0/` contains the best weights, an inference model, training history, and timings.
When enabled, test evaluation includes regional metric tables, per-scan inference timings,
and original-grid ROI/brain predictions. Status columns distinguish unavailable annotations
from annotated empty structures; inspect valid counts before summarizing results.

For the conventional U-Net baseline, set `architecture="unet"`, `residual=False`,
and `attention=False`, and update the experiment name while
reusing the same saved split. Keep other settings fixed for a controlled comparison.
This notebook does not perform ablations, statistical comparisons, fine-tuning, or sensitivity analysis.
See `docs/experiments.md` for the core API and current limitations.

## Explore test predictions against ground truth

Run the cell below after training/evaluation, or point `VIEW_RUN` at an existing matching run folder to inspect it without fitting. Saved full-volume `regions.nii.gz` files are loaded directly; no predictions are recomputed. This requires inference and prediction saving to have been enabled.

The table lists every available scan/timepoint and fold. The viewer initially loads up to three rows; set `VIEW_SCAN_INDICES=(0, 4)` to explore other rows. Use the Plotly dropdown to select a scan and reference-oriented plane, then drag the slice slider. Zoom/pan works in each MRI/GT/prediction panel.

Both overlays use the same class colors. The title lists unannotated GT regions: their absence is not evidence of a prediction error. Set `VIEW_REGION` to a foreground **original annotation ID** (for example 13 for the ipsilateral ventricle) to isolate one region. `VIEW_OPACITY` controls the overlay. Previews sample up to 64 slices per plane and 192 pixels per side; set `VIEW_MAX_SLICES=None` for every slice and raise `VIEW_MAX_PIXELS` for more detail. Saved masks are unchanged. The controls use native Plotly rendering.


In [ ]:
from src.evaluation.inference import saved_prediction_manifest, prediction_comparison_figure

VIEW_RUN: Path | None = None  # Existing run folder, or use the result above.
VIEW_SCAN_INDICES: tuple[int, ...] | None = None  # None: first three manifest rows.
VIEW_REGION: int | None = None  # Original annotation ID (e.g. 13 = ipsi ventricle); None shows all.
VIEW_OPACITY: float = 0.55
VIEW_MAX_SLICES: int | None = 64
VIEW_MAX_PIXELS: int = 192
VIEW_ROLE: str = 'test'

viewer_role = VIEW_ROLE
viewer_run = VIEW_RUN
current_result = globals().get("result")
if viewer_run is None and current_result is not None:
    viewer_run = current_result.run_dir
if viewer_run is None and RESUME_RUN is not None:
    viewer_run = RESUME_RUN

if viewer_run is None:
    print("Train/evaluate first, or set VIEW_RUN to an existing run folder.")
else:
    prediction_views = saved_prediction_manifest(prepared, viewer_run, role=viewer_role)
    display(prediction_views)

    if prediction_views.empty:
        print("No saved predictions for this role. Check the run, VIEW_ROLE and prediction-saving settings.")
    else:
        prediction_figure = prediction_comparison_figure(
            data=prepared, predictions=prediction_views,
            scan_indices=VIEW_SCAN_INDICES, region=VIEW_REGION,
            opacity=VIEW_OPACITY, max_slices=VIEW_MAX_SLICES,
            max_pixels=VIEW_MAX_PIXELS,
        )
        prediction_figure.show(renderer="vscode")


## Evaluate the best checkpoint after stopping training

Interrupt training, then run this cell independently of `result`. Set `CHECKPOINT_RUN` to the existing **run folder**, select `CHECKPOINT_FOLD`, and enable `RUN_CHECKPOINT_EVALUATION`. After a kernel restart, only the repository setup cell and this cell are needed: the saved experiment supplies architecture, preprocessing, label policy and animal selection.

The default cohort matches this notebook. Fixed-split runs use test animals, CV uses outer holdouts, and historical rat CV uses validation animals. Every selected animal includes all timepoints. A best checkpoint must already exist; an unfinished epoch is not evaluated. The committed selected-weight copy is used when recovery state is available.

This performs full-volume inference only, saves metrics/predictions to a fresh folder outside the training run, and opens the MRI/GT/prediction viewer. Training weights, optimizer, history and completion flags are left intact. Inference can take time; leave the switch False to skip it. Use notebook 09 to reopen an evaluation without recomputing predictions.

The common RARE reference is explicitly used for checkpoint evaluation, including older checkpoints. A model trained with a different orientation receives a changed input convention; those scores should be interpreted as evaluation with an orientation override. Training recovery files remain unchanged.


The independent cell exposes `checkpoint_postprocessing` and always restores original annotation IDs, including when the saved run predates these settings. It writes `prediction_labels.json`, so the viewer uses the exact exported IDs. `CHECKPOINT_REGION` selects an original annotation ID. Changing evaluation settings does not change training targets or weights.


In [ ]:
from src.experiments.config import PostprocessingConfig
from src.evaluation.inference import (
    best_checkpoint_settings, evaluate_saved_model,
    saved_prediction_manifest, prediction_comparison_figure,
)

RUN_CHECKPOINT_EVALUATION: bool = False
CHECKPOINT_RUN: Path | None = None  # results/<model>/<run>; required after interruption/restart.
CHECKPOINT_FOLD: int = 0
CHECKPOINT_OUTPUT: Path | None = None  # None creates a fresh evaluation folder.
CHECKPOINT_SCAN_INDICES: tuple[int, ...] | None = None
CHECKPOINT_REGION: int | None = None
CHECKPOINT_MAX_SLICES: int | None = 64
CHECKPOINT_ROLE: str = 'test'

checkpoint_run = CHECKPOINT_RUN
current_result = globals().get("result")
if checkpoint_run is None and current_result is not None:
    checkpoint_run = current_result.run_dir
if checkpoint_run is None:
    checkpoint_run = globals().get("RESUME_RUN")

checkpoint_postprocessing = PostprocessingConfig(
    clean_brain_mask=True,
    fill_small_holes=True, holes_max_area=20000,
    remove_small_objects=True, object_min_area=30000,
    split_ventricles=True, use_centroids=False,
    default_hemisphere="right", lesion_voxels_threshold=10,
)

checkpoint_evaluation = None
if RUN_CHECKPOINT_EVALUATION:
    if checkpoint_run is None:
        raise ValueError("Set CHECKPOINT_RUN to the stopped run folder first")
    checkpoint_settings = best_checkpoint_settings(
        run_dir=checkpoint_run, fold=CHECKPOINT_FOLD, role=CHECKPOINT_ROLE,
        output_dir=CHECKPOINT_OUTPUT, reference=REFERENCE,
        original_label_ids=True, postprocessing=checkpoint_postprocessing,
    )
    print("Selected weights:", checkpoint_settings.model_file)
    checkpoint_evaluation = evaluate_saved_model(checkpoint_settings)
    print("Evaluation saved to:", checkpoint_evaluation.output_dir)
    checkpoint_metrics = checkpoint_evaluation.metrics
    checkpoint_dice = checkpoint_metrics.loc[checkpoint_metrics["Metric"] == "Dice"]
    display(checkpoint_dice)
    checkpoint_views = saved_prediction_manifest(
        data=checkpoint_evaluation.data, run_dir=checkpoint_evaluation.output_dir,
        role=CHECKPOINT_ROLE,
    )
    display(checkpoint_views)
    checkpoint_figure = prediction_comparison_figure(
        data=checkpoint_evaluation.data, predictions=checkpoint_views,
        scan_indices=CHECKPOINT_SCAN_INDICES, region=CHECKPOINT_REGION,
        max_slices=CHECKPOINT_MAX_SLICES,
    )
    checkpoint_figure.show(renderer="vscode")
else:
    print("Set CHECKPOINT_RUN and RUN_CHECKPOINT_EVALUATION=True to evaluate saved best weights.")
